<a href="https://colab.research.google.com/github/Shareencoco/ML-Miniproject/blob/main/HALE_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the dataset
file_path = ('/content/sample_data/merge.csv')
data = pd.read_csv(file_path, encoding='ISO-8859-1')

In [ ]:
missing_values = data.isnull().sum()
print("Missing values:")
print(missing_values)

Missing values:
Period                                                                                                                                                        0
ParentLocation                                                                                                                                                0
Location                                                                                                                                                      0
Dim1                                                                                                                                                          0
HALE_Birth                                                                                                                                                    0
HALE_60                                                                                                                                                       0
infant mortality rate (b

In [ ]:
#Filter the data for the period 2000-2019
data_filtered = data[(data['Period'] >= 2000) & (data['Period'] <= 2019)]
#Drop columns with more than 60% missing values
threshold = 0.6 * len(data_filtered)  # 60% threshold
data_filtered_cleaned = data_filtered.dropna(thresh=threshold, axis=1)

In [ ]:
#remaining missing values
missing_values = data_filtered_cleaned.isnull().sum()
print("Missing values after filter:")
print(missing_values)

Missing values after filter:
Period                                                                                                                                                       0
ParentLocation                                                                                                                                               0
Location                                                                                                                                                     0
Dim1                                                                                                                                                         0
HALE_Birth                                                                                                                                                   0
HALE_60                                                                                                                                                      0
infant mortality 

In [ ]:
# Backward fill for missing values ONLY in the year 2000
# Isolate rows for 2000 and apply backward fill for 2000 using future data from the same country
for country in data_filtered_cleaned['Location'].unique():
    country_data = data_filtered_cleaned[data_filtered_cleaned['Location'] == country]
    country_2000_data = country_data[country_data['Period'] == 2000]

    if country_2000_data.isnull().values.any():  # Check if there are missing values
        # Get the next available data after 2000 for this country
        future_data = country_data[country_data['Period'] > 2000].sort_values('Period')

        # Fill missing values in 2000 with the first available future data
        if not future_data.empty:
            data_filtered_cleaned.loc[(data_filtered_cleaned['Location'] == country) &
                                      (data_filtered_cleaned['Period'] == 2000)] = (
                data_filtered_cleaned.loc[(data_filtered_cleaned['Location'] == country) &
                                          (data_filtered_cleaned['Period'] == 2000)].fillna(future_data.iloc[0])
            )
# Round values after backward fill to ensure consistency
data_filtered_cleaned = data_filtered_cleaned.round(2)

In [ ]:
#  Check for remaining missing values
missing_values = data_filtered_cleaned.isnull().sum()
print("Missing values after backward filling:")
print(missing_values)

Missing values after backward filling:
Period                                                                                                                                                       0
ParentLocation                                                                                                                                               0
Location                                                                                                                                                     0
Dim1                                                                                                                                                         0
HALE_Birth                                                                                                                                                   0
HALE_60                                                                                                                                                      0
infant 

In [ ]:
# Apply interpolation for the remaining missing values
data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))

# Round values after interpolation
data_interpolated = data_interpolated.round(2)


/tmp/ipykernel_2169/3489596845.py:2: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))
/tmp/ipykernel_2169/3489596845.py:2: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))
/tmp/ipykernel_2169/3489596845.py:2: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  data_interpolated = data_filtered_cleaned.groupby('Location').apply(lambda group: group.interpolate(method='linear'))
/tmp/ipykernel_2169

In [ ]:
# Check for remaining missing values
missing_values = data_interpolated.isnull().sum()
print("Missing values after interpolation:")
print(missing_values)

Missing values after interpolation:
Period                                                                                                                                                       0
ParentLocation                                                                                                                                               0
Location                                                                                                                                                     0
Dim1                                                                                                                                                         0
HALE_Birth                                                                                                                                                   0
HALE_60                                                                                                                                                      0
infant mor

In [ ]:
# Apply global mean imputation for any remaining missing values
data_filled = data_interpolated.fillna(data_interpolated.mean(numeric_only=True))

#Round values after global mean imputation
data_filled_rounded = data_filled.round(2)

In [ ]:
# Check for remaining missing values
missing_values = data_filled_rounded.isnull().sum()
print("Missing values after global mean imputation:")
print(missing_values)

Missing values after global mean imputation:
Period                                                                                                                                                     0
ParentLocation                                                                                                                                             0
Location                                                                                                                                                   0
Dim1                                                                                                                                                       0
HALE_Birth                                                                                                                                                 0
HALE_60                                                                                                                                                    0
infant mortal

In [ ]:
import os

# Define the directory path
directory_path = '/data/raw'

# Create the directory if it does not exist
if not os.path.exists(directory_path):
    os.makedirs(directory_path)
    print(f"Directory '{directory_path}' created.")

# Save the cleaned dataset
data_filled_rounded.to_csv(os.path.join(directory_path, 'cleaned_dataset.csv'), index=False)

print("Data preprocessing complete! The cleaned dataset with rounded values is saved as 'cleaned_dataset.csv'.")

Directory '/data/raw' created.
Data preprocessing complete! The cleaned dataset with rounded values is saved as 'cleaned_dataset.csv'.
